# Setup and Imports

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import joblib # For saving/loading models and preprocessors
import matplotlib.pyplot as plt # For regression plot
import seaborn as sns # For general plotting if needed

# Scikit-learn imports for modeling and preprocessing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score,
    classification_report, confusion_matrix
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer # Still useful for creating custom preprocessors

# Get the current working directory of the notebook
notebook_dir = os.getcwd()

# Assuming your 'src' folder is one level up from the notebook's directory
project_root = os.path.abspath(os.path.join(notebook_dir, '..'))

# Add the project root to sys.path to allow importing from src
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    print(f"Added project root '{project_root}' to sys.path.")
else:
    print(f"Project root '{project_root}' was already in sys.path.")

# Import constants and data preparation functions from src
try:
    from src.config import (
        RAW_DATA_PATH, PROCESSED_DATA_PATH, POLICY_ID_COL, TRANSACTION_MONTH_COL,
        TOTAL_CLAIMS_COL, TOTAL_PREMIUM_COL, HAS_CLAIM_COL, # HAS_CLAIM_COL now exists directly
        CLAIM_PROBABILITY_TARGET, CLAIM_SEVERITY_TARGET, RANDOM_STATE
    )
    from src.data_preparation import load_raw_data, prepare_data_for_modeling
    print("Successfully imported modules from src.")

    # Placeholder for initial_data_cleaning if it's not in your src/data_preparation.py
    # This function should ideally handle things like duplicate removal and basic type cleaning.
    def initial_data_cleaning(df: pd.DataFrame) -> pd.DataFrame:
        """
        Performs basic initial cleaning steps not handled by prepare_data_for_modeling's internal steps.
        E.g., duplicate removal.
        """
        print("Running placeholder initial_data_cleaning...")
        df_cleaned = df.drop_duplicates().copy()
        # Ensure column names are stripped of whitespace if any
        df_cleaned.columns = df_cleaned.columns.str.strip()
        print(f"Removed duplicates. New shape: {df_cleaned.shape}")
        return df_cleaned

except ModuleNotFoundError as e:
    print(f"Error importing src modules: {e}")
    print("Please ensure your 'src' folder is correctly set up and 'src/config.py', 'src/data_preparation.py' exist.")

# Set display options for pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:.4f}'.format) # More precision for model metrics

print("Setup and imports complete.")

Added project root 'C:\Users\benke\Desktop\DataScience\10Academy\Kefiya_AI_Mastery\Week3\KAIM_W3_Insurance_risk_analysis\KAIM_W03_AlphaCare_Insurance_Risk_Analysis' to sys.path.
Successfully imported modules from src.
Setup and imports complete.


# Load Raw Data & Perform Initial Cleaning

In [3]:
print("Loading raw data...")
df_raw = load_raw_data(RAW_DATA_PATH)

if df_raw.empty:
    print("Failed to load raw data. Please check RAW_DATA_PATH in src/config.py and file existence.")
else:
    print(f"Raw data loaded successfully from '{RAW_DATA_PATH}'. Shape: {df_raw.shape}")
    print("\nPerforming initial data cleaning...")
    # This initial_data_cleaning handles things like duplicate removal and stripping column names
    df_cleaned = initial_data_cleaning(df_raw)
    print(f"Initial data cleaning complete. Cleaned data shape: {df_cleaned.shape}")
    print("\nDisplaying info of cleaned data (first 5 rows):")
    display(df_cleaned.head())

Loading raw data...


C:\Users\benke\Desktop\DataScience\10Academy\Kefiya_AI_Mastery\Week3\KAIM_W3_Insurance_risk_analysis\KAIM_W03_AlphaCare_Insurance_Risk_Analysis\src\data_preparation.py:25: DtypeWarning: Columns (32,37) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, sep='|')


Data loaded successfully from C:\Users\benke\Desktop\DataScience\10Academy\Kefiya_AI_Mastery\Week3\KAIM_W3_Insurance_risk_analysis\KAIM_W03_AlphaCare_Insurance_Risk_Analysis\data\raw\MachineLearningRating_v3.txt. Shape: (1000098, 52)
Raw data loaded successfully from 'C:\Users\benke\Desktop\DataScience\10Academy\Kefiya_AI_Mastery\Week3\KAIM_W3_Insurance_risk_analysis\KAIM_W03_AlphaCare_Insurance_Risk_Analysis\data\raw\MachineLearningRating_v3.txt'. Shape: (1000098, 52)

Performing initial data cleaning...
Running placeholder initial_data_cleaning...
Removed duplicates. New shape: (1000098, 52)
Initial data cleaning complete. Cleaned data shape: (1000098, 52)

Displaying info of cleaned data (first 5 rows):


,UnderwrittenCoverID,PolicyID,TransactionMonth,IsVATRegistered,Citizenship,LegalType,Title,Language,Bank,AccountType,MaritalStatus,Gender,Country,Province,PostalCode,MainCrestaZone,SubCrestaZone,ItemType,mmcode,VehicleType,RegistrationYear,make,Model,Cylinders,cubiccapacity,kilowatts,bodytype,NumberOfDoors,VehicleIntroDate,CustomValueEstimate,AlarmImmobiliser,TrackingDevice,CapitalOutstanding,NewVehicle,WrittenOff,Rebuilt,Converted,CrossBorder,NumberOfVehiclesInFleet,SumInsured,TermFrequency,CalculatedPremiumPerTerm,ExcessSelected,CoverCategory,CoverType,CoverGroup,Section,Product,StatutoryClass,StatutoryRiskType,TotalPremium,TotalClaims
0,145249,12827,2015-03-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,Not specified,Not specified,South Africa,Gauteng,1459,Rand East,Rand East,Mobility - Motor,44069150.0000,Passenger Vehicle,2004,MERCEDES-BENZ,E 240,6.0000,2597.0000,130.0000,S/D,4.0000,6/2002,119300.0000,Yes,No,119300,More than 6 months,NaN,NaN,NaN,NaN,NaN,0.0100,Monthly,25.0000,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.9298,0.0000
1,145249,12827,2015-05-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,Not specified,Not specified,South Africa,Gauteng,1459,Rand East,Rand East,Mobility - Motor,44069150.0000,Passenger Vehicle,2004,MERCEDES-BENZ,E 240,6.0000,2597.0000,130.0000,S/D,4.0000,6/2002,119300.0000,Yes,No,119300,More than 6 months,NaN,NaN,NaN,NaN,NaN,0.0100,Monthly,25.0000,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.9298,0.0000
2,145249,12827,2015-07-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,Not specified,Not specified,South Africa,Gauteng,1459,Rand East,Rand East,Mobility - Motor,44069150.0000,Passenger Vehicle,2004,MERCEDES-BENZ,E 240,6.0000,2597.0000,130.0000,S/D,4.0000,6/2002,119300.0000,Yes,No,119300,More than 6 months,NaN,NaN,NaN,NaN,NaN,0.0100,Monthly,25.0000,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.0000,0.0000
3,145255,12827,2015-05-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,Not specified,Not specified,South Africa,Gauteng,1459,Rand East,Rand East,Mobility - Motor,44069150.0000,Passenger Vehicle,2004,MERCEDES-BENZ,E 240,6.0000,2597.0000,130.0000,S/D,4.0000,6/2002,119300.0000,Yes,No,119300,More than 6 months,NaN,NaN,NaN,NaN,NaN,119300.0000,Monthly,584.6468,Mobility - Metered Taxis - R2000,Own damage,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,512.8481,0.0000
4,145255,12827,2015-07-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,Not specified,Not specified,South Africa,Gauteng,1459,Rand East,Rand East,Mobility - Motor,44069150.0000,Passenger Vehicle,2004,MERCEDES-BENZ,E 240,6.0000,2597.0000,130.0000,S/D,4.0000,6/2002,119300.0000,Yes,No,119300,More than 6 months,NaN,NaN,NaN,NaN,NaN,119300.0000,Monthly,584.6468,Mobility - Metered Taxis - R2000,Own damage,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.0000,0.0000


In [2]:
#Define Features and Prepare Data for Modeling (Classification & Regression)

In [4]:
print("\n--- Defining Features and Preparing Data for Modeling ---")

# Define columns by their type for the prepare_data_for_modeling function
# These lists are updated based on your provided actual column names.

# Removed: 'Age', 'VehicleAge', 'Margin', 'VehiclePower', 'GeoCode'
# Added: other actual numerical columns
numerical_cols_initial = [
    'RegistrationYear', 'Cylinders', 'cubiccapacity', 'kilowatts',
    'NumberOfDoors', 'CustomValueEstimate', 'CapitalOutstanding',
    'NumberOfVehiclesInFleet', 'SumInsured', 'CalculatedPremiumPerTerm',
    'ExcessSelected', TOTAL_PREMIUM_COL # TOTAL_PREMIUM_COL should map to 'TotalPremium'
]

# Changed 'DriverGender' to 'Gender'
# Added many other actual categorical columns
categorical_cols_initial = [
    'IsVATRegistered', 'Citizenship', 'LegalType', 'Title', 'Language',
    'Bank', 'AccountType', 'MaritalStatus', 'Gender', 'Country', 'Province',
    'PostalCode', 'MainCrestaZone', 'SubCrestaZone', 'ItemType', 'mmcode',
    'VehicleType', 'make', 'Model', 'bodytype', 'AlarmImmobiliser',
    'TrackingDevice', 'NewVehicle', 'WrittenOff', 'Rebuilt', 'Converted',
    'CrossBorder', 'TermFrequency', 'CoverCategory', 'CoverType',
    'CoverGroup', 'Section', 'Product', 'StatutoryClass', 'StatutoryRiskType'
]

date_cols_initial = [TRANSACTION_MONTH_COL, 'VehicleIntroDate']

# Add 'UnderwrittenCoverID' to drop, as it's likely another ID
cols_to_drop_initial = [POLICY_ID_COL, 'UnderwrittenCoverID']

# --- 3.1: Prepare Data for Classification Task (Claim Probability) ---
print("\nPreparing data for Classification Task (untransformed X, unfitted preprocessor)...")
X_clf_unprocessed, y_classification, preprocessor_clf_unfitted = prepare_data_for_modeling(
    df=df_cleaned.copy(), # Pass a copy to avoid modifying original df_cleaned
    numerical_cols=numerical_cols_initial,
    categorical_cols=categorical_cols_initial,
    date_cols=date_cols_initial,
    cols_to_drop=cols_to_drop_initial,
    problem_type='classification'
)
print(f"Classification features (X_clf_unprocessed) shape: {X_clf_unprocessed.shape}")
print(f"Classification target (y_classification) shape: {y_classification.shape}")


# --- 3.2: Prepare Data for Regression Task (Claim Severity) ---
print("\nPreparing data for Regression Task (untransformed X, unfitted preprocessor)...")
X_reg_unprocessed, y_regression, preprocessor_reg_unfitted = prepare_data_for_modeling(
    df=df_cleaned.copy(), # Pass a copy
    numerical_cols=numerical_cols_initial,
    categorical_cols=categorical_cols_initial,
    date_cols=date_cols_initial,
    cols_to_drop=cols_to_drop_initial,
    problem_type='regression' # This will filter for claims > 0 and handle targets
)
print(f"Regression features (X_reg_unprocessed) shape: {X_reg_unprocessed.shape}")
print(f"Regression target (y_regression) shape: {y_regression.shape}")

print("\nInitial data preparation for modeling complete. Preprocessors are defined but not yet fitted.")


--- Defining Features and Preparing Data for Modeling ---

Preparing data for Classification Task (untransformed X, unfitted preprocessor)...


C:\Users\benke\Desktop\DataScience\10Academy\Kefiya_AI_Mastery\Week3\KAIM_W3_Insurance_risk_analysis\KAIM_W03_AlphaCare_Insurance_Risk_Analysis\src\data_preparation.py:62: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[date_col] = pd.to_datetime(df[date_col], errors='coerce')


Classification features (X_clf_unprocessed) shape: (1000098, 51)
Classification target (y_classification) shape: (1000098,)

Preparing data for Regression Task (untransformed X, unfitted preprocessor)...


C:\Users\benke\Desktop\DataScience\10Academy\Kefiya_AI_Mastery\Week3\KAIM_W3_Insurance_risk_analysis\KAIM_W03_AlphaCare_Insurance_Risk_Analysis\src\data_preparation.py:62: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[date_col] = pd.to_datetime(df[date_col], errors='coerce')


Regression features (X_reg_unprocessed) shape: (2788, 51)
Regression target (y_regression) shape: (2788,)

Initial data preparation for modeling complete. Preprocessors are defined but not yet fitted.


# Train-Test Split for Classification and Regression

In [5]:
print("\n--- Performing Train-Test Split ---")

# --- 4.1: Split data for Classification Task ---
print("\nSplitting data for Classification Task (Claim Probability)...")
X_cls_train, X_cls_test, y_cls_train, y_cls_test = train_test_split(
    X_clf_unprocessed, y_classification, test_size=0.2, random_state=RANDOM_STATE, stratify=y_classification
)
print(f"Classification Train Set: X_cls_train {X_cls_train.shape}, y_cls_train {y_cls_train.shape}")
print(f"Classification Test Set: X_cls_test {X_cls_test.shape}, y_cls_test {y_cls_test.shape}")
print(f"y_cls_train value counts:\n{y_cls_train.value_counts(normalize=True)}")
print(f"y_cls_test value counts:\n{y_cls_test.value_counts(normalize=True)}")


# --- 4.2: Split data for Regression Task ---
print("\nSplitting data for Regression Task (Claim Severity, for claims > 0)...")
# Note: y_regression is already filtered for claims > 0 by prepare_data_for_modeling
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg_unprocessed, y_regression, test_size=0.2, random_state=RANDOM_STATE
)
print(f"Regression Train Set: X_reg_train {X_reg_train.shape}, y_reg_train {y_reg_train.shape}")
print(f"Regression Test Set: X_reg_test {X_reg_test.shape}, y_reg_test {y_reg_test.shape}")

print("\nTrain-Test Split complete for both tasks.")


--- Performing Train-Test Split ---

Splitting data for Classification Task (Claim Probability)...
Classification Train Set: X_cls_train (800078, 51), y_cls_train (800078,)
Classification Test Set: X_cls_test (200020, 51), y_cls_test (200020,)
y_cls_train value counts:
HasClaim
0   0.9972
1   0.0028
Name: proportion, dtype: float64
y_cls_test value counts:
HasClaim
0   0.9972
1   0.0028
Name: proportion, dtype: float64

Splitting data for Regression Task (Claim Severity, for claims > 0)...
Regression Train Set: X_reg_train (2230, 51), y_reg_train (2230,)
Regression Test Set: X_reg_test (558, 51), y_reg_test (558,)

Train-Test Split complete for both tasks.


# Classification Model Training (Claim Probability - HasClaim)

In [8]:
# print("\n--- Training Classification Model (Claim Probability) ---")

# Create a full pipeline that includes preprocessing and the classifier
# This ensures preprocessing is applied consistently and prevents data leakage
classification_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_clf_unfitted), # This preprocessor will be fitted on X_cls_train
    ('classifier', RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=200, class_weight='balanced'))
])

# Train the pipeline
print(f"Training Classification Pipeline (RandomForestClassifier with preprocessing)...")
classification_pipeline.fit(X_cls_train, y_cls_train)
print("Classification model training complete.")

# Make predictions on the test set using the trained pipeline
y_cls_pred = classification_pipeline.predict(X_cls_test)
y_cls_pred_proba = classification_pipeline.predict_proba(X_cls_test)[:, 1] # Probability of the positive class

# Evaluate Classification Model
print("\n--- Classification Model Evaluation ---")
print("Accuracy:", accuracy_score(y_cls_test, y_cls_pred))
print("Precision (Positive Class):", precision_score(y_cls_test, y_cls_pred))
print("Recall (Positive Class):", recall_score(y_cls_test, y_cls_pred))
print("F1-Score (Positive Class):", f1_score(y_cls_test, y_cls_pred))
print("ROC AUC Score:", roc_auc_score(y_cls_test, y_cls_pred_proba))

print("\nClassification Report:")
print(classification_report(y_cls_test, y_cls_pred))

print("\nConfusion Matrix:")
# Rows: Actual, Columns: Predicted
# [[True Negatives, False Positives], [False Negatives, True Positives]]
print(confusion_matrix(y_cls_test, y_cls_pred))

# Save the trained classification pipeline
cls_pipeline_path = os.path.join(project_root, 'models', 'classification_pipeline.joblib')
os.makedirs(os.path.dirname(cls_pipeline_path), exist_ok=True)
joblib.dump(classification_pipeline, cls_pipeline_path)
print(f"\nClassification pipeline saved to: {cls_pipeline_path}")

Training Classification Pipeline (RandomForestClassifier with preprocessing)...


ValueError: Cannot use mean strategy with non-numeric data:
could not convert string to float: '285700,00'

# Regression Model Training (Claim Severity - TotalClaims for claims > 0)

In [ ]:
print("\n--- Training Regression Model (Claim Severity) ---")

# Create a full pipeline that includes preprocessing and the regressor
# This ensures preprocessing is applied consistently and prevents data leakage
regression_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_reg_unfitted), # This preprocessor will be fitted on X_reg_train
    ('regressor', RandomForestRegressor(random_state=RANDOM_STATE, n_estimators=200))
])

# Train the pipeline
if not y_reg_train.empty:
    print(f"Training Regression Pipeline (RandomForestRegressor with preprocessing)...")
    regression_pipeline.fit(X_reg_train, y_reg_train)
    print("Regression model training complete.")

    # Make predictions on the test set using the trained pipeline
    y_reg_pred = regression_pipeline.predict(X_reg_test)

    # Evaluate Regression Model
    print("\n--- Regression Model Evaluation ---")
    print("Mean Absolute Error (MAE):", mean_absolute_error(y_reg_test, y_reg_pred))
    print("Mean Squared Error (MSE):", mean_squared_error(y_reg_test, y_reg_pred))
    print("Root Mean Squared Error (RMSE):", np.sqrt(mean_squared_error(y_reg_test, y_reg_pred)))
    print("R2 Score:", r2_score(y_reg_test, y_reg_pred))

    # Optional: Visualizing Regression Predictions (Actual vs. Predicted)
    plt.figure(figsize=(10, 6))
    plt.scatter(y_reg_test, y_reg_pred, alpha=0.3)
    # Add a diagonal line for perfect predictions
    max_val = max(y_reg_test.max(), y_reg_pred.max())
    min_val = min(y_reg_test.min(), y_reg_pred.min())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
    plt.xlabel(f"Actual {TOTAL_CLAIMS_COL}")
    plt.ylabel(f"Predicted {TOTAL_CLAIMS_COL}")
    plt.title("Actual vs. Predicted Claim Amounts (Regression)")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Save the trained regression pipeline
    reg_pipeline_path = os.path.join(project_root, 'models', 'regression_pipeline.joblib')
    os.makedirs(os.path.dirname(reg_pipeline_path), exist_ok=True)
    joblib.dump(regression_pipeline, reg_pipeline_path)
    print(f"\nRegression pipeline saved to: {reg_pipeline_path}")
else:
    print("No data available for regression training (y_reg_train is empty). Skipping regression model training.")

print("\n--- Model Development and Training Complete ---")